In [1]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import awkward as ak
from matplotlib.patches import Circle, Patch
from scipy.optimize import curve_fit
from collections import defaultdict

# ============================================================
# THESIS-STYLE PLOT SETTINGS
# ============================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 16,
    "axes.titlesize": 18,
    "legend.fontsize": 11,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "axes.linewidth": 1.2,
    "lines.linewidth": 1.8,
    "figure.figsize": (12.8, 6.2),
    "figure.dpi": 140,
})

# ============================================================
# PDG CODES
# ============================================================
DEUTERON_PDG = 1000010020
GAMMA_PDG = 22
ELECTRON_PDG = 11
NEUTRON_PDG = 2112

# ============================================================
# LOAD DATA (once, flattened)
# ============================================================
path = "Sim_D2ODetector003.root"   # <-- change to your file
file = uproot.open(path)
event_data = file["Sim_Tree"]["eventData"]

all_pdg = event_data["allParticlePDG"].array()
all_track = event_data["allParticleTrackID"].array()
all_parent = event_data["allParticleParentID"].array()
all_time = event_data["allParticleTime"].array()          # ns
all_energy = event_data["allParticleEnergy"].array()      # MeV
all_posX = event_data["allParticlePosX"].array()
all_posY = event_data["allParticlePosY"].array()
all_posZ = event_data["allParticlePosZ"].array()

# Flatten
pdg = ak.to_numpy(ak.flatten(all_pdg))
track = ak.to_numpy(ak.flatten(all_track))
parent = ak.to_numpy(ak.flatten(all_parent))
time = ak.to_numpy(ak.flatten(all_time))
energy = ak.to_numpy(ak.flatten(all_energy))
posX = ak.to_numpy(ak.flatten(all_posX))
posY = ak.to_numpy(ak.flatten(all_posY))
posZ = ak.to_numpy(ak.flatten(all_posZ))

print(f"Total particles: {len(pdg):,}")

# ============================================================
# BUILD LOOKUP DICTIONARIES (fast O(N))
# ============================================================
print("Building lookup tables...")

# Map parent track ID -> list of child indices (for gammas and electrons separately)
gamma_children = defaultdict(list)      # neutron track ID -> gamma indices
electron_children = defaultdict(list)   # gamma track ID -> electron indices

# Also store deuteron info for quick access
deuteron_indices = []
deuteron_parents = []

for i, (p, tr, par) in enumerate(zip(pdg, track, parent)):
    if p == DEUTERON_PDG:
        deuteron_indices.append(i)
        deuteron_parents.append(par)
    elif p == GAMMA_PDG:
        gamma_children[par].append(i)      # par is the neutron that created this gamma
    elif p == ELECTRON_PDG:
        electron_children[par].append(i)   # par is the gamma that created this electron

print(f"Deuterons found: {len(deuteron_indices):,}")
print(f"Gamma parents (neutrons): {len(gamma_children):,}")
print(f"Electron parents (gammas): {len(electron_children):,}")

# ============================================================
# EXTRACT CAPTURE CHAIN: neutron -> gamma -> electron
# ============================================================
print("Extracting capture electrons...")

capture_times = []       # μs
capture_energies = []    # MeV
capture_positions = []   # (x,y,z) in cm

for d_idx, n_track in zip(deuteron_indices, deuteron_parents):
    # Get gammas from this neutron
    gamma_idxs = gamma_children.get(n_track, [])
    for g_idx in gamma_idxs:
        gamma_track = track[g_idx]
        # Get electrons from this gamma
        electron_idxs = electron_children.get(gamma_track, [])
        for e_idx in electron_idxs:
            e_time_ns = time[e_idx]
            e_energy = energy[e_idx]
            # Only Compton electrons (energy > 0.1 MeV)
            if e_energy > 0.1:
                capture_times.append(e_time_ns / 1000.0)   # ns -> μs
                capture_energies.append(e_energy)
                capture_positions.append((posX[e_idx], posY[e_idx], posZ[e_idx]))

capture_times = np.array(capture_times)
capture_energies = np.array(capture_energies)
capture_positions = np.array(capture_positions)

print(f"Capture electrons found: {len(capture_times):,}")

if len(capture_times) == 0:
    raise SystemExit("No capture electrons found. Check simulation or thresholds.")

# ============================================================
# PLOTTING (as before, but with the new data)
# ============================================================

def exp_decay(t, A, tau, C):
    return A * np.exp(-t / tau) + C

# ---- Figure 1: XY positions ----
fig, ax = plt.subplots()
sc = ax.scatter(capture_positions[:,0], capture_positions[:,1],
                c=capture_times, cmap='viridis', s=15, alpha=0.7,
                edgecolors='black', linewidth=0.3)
boundaries = [(33.5, 'D₂O', 'blue'), (42.5, 'Acrylic', 'magenta'),
              (48.5, 'Tyvek', 'gray'), (52.5, 'Steel', 'darkgray')]
for r, label, col in boundaries:
    circle = Circle((0,0), r, fill=False, edgecolor=col, linestyle='--', linewidth=1.5)
    ax.add_patch(circle)
ax.set_xlabel('X (cm)'); ax.set_ylabel('Y (cm)')
ax.set_title('Capture Positions (XY)')
ax.set_aspect('equal'); ax.set_xlim(-60,60); ax.set_ylim(-60,60)
cbar = plt.colorbar(sc); cbar.set_label('Time (μs)')
legend_elements = [Patch(facecolor='none', edgecolor=c, linestyle='--', label=l)
                   for r, l, c in boundaries]
ax.legend(handles=legend_elements, loc='upper right')
ax.grid(True, linestyle='--', alpha=0.4); ax.minorticks_on()
plt.tight_layout(); plt.savefig('fig1_capture_positions.png', dpi=300); plt.show()

# ---- Figure 2: Time distribution with fit ----
fig, ax = plt.subplots()
bins = np.linspace(0, 500, 100)
hist, edges = np.histogram(capture_times, bins=bins)
centers = (edges[:-1] + edges[1:]) / 2
width = edges[1] - edges[0]
norm = hist / width

ax.step(centers, norm, where='mid', color='#1f77b4', linewidth=2, label='Data')
ax.plot(centers, norm, 'o', color='#1f77b4', markersize=3,
        markerfacecolor='white', markeredgecolor='#1f77b4')

mask = (centers > 50) & (norm > 0)
if np.sum(mask) > 5:
    popt, _ = curve_fit(exp_decay, centers[mask], norm[mask], p0=[max(norm), 100, 0])
    x_fit = np.linspace(0, 500, 200)
    ax.plot(x_fit, exp_decay(x_fit, *popt), 'r-', linewidth=2,
            label=f'Fit: τ = {popt[1]:.1f} μs')
    print(f"Fit result: τ = {popt[1]:.1f} μs")

ax.axvline(np.mean(capture_times), color='red', linestyle='--', alpha=0.7,
           label=f'Mean: {np.mean(capture_times):.1f} μs')
ax.set_xlabel('Capture Time (μs)'); ax.set_ylabel(f'Counts / {width:.1f} μs')
ax.set_title('Capture Time Distribution')
ax.set_xlim(0,500); ax.set_yscale('log')
ax.legend(); ax.grid(True, linestyle='--', alpha=0.4); ax.minorticks_on()
plt.tight_layout(); plt.savefig('fig2_capture_time_fit.png', dpi=300); plt.show()

# ---- Figure 3: Deuteron energy spectrum (optional, using stored deuteron energies) ----
# We can compute deuteron energies from the indices we stored
deuteron_energies_keV = energy[deuteron_indices] * 1000
fig, ax = plt.subplots()
bins_e = np.linspace(0, 2.0, 50)
hist_e, edges_e = np.histogram(deuteron_energies_keV, bins=bins_e)
centers_e = (edges_e[:-1] + edges_e[1:]) / 2
width_e = edges_e[1] - edges_e[0]
norm_e = hist_e / width_e
ax.step(centers_e, norm_e, where='mid', color='#9467bd', linewidth=2, label='Data')
ax.plot(centers_e, norm_e, 'o', color='#9467bd', markersize=3,
        markerfacecolor='white', markeredgecolor='#9467bd')
ax.axvline(1.3, color='red', linestyle='--', linewidth=2, label='Expected (1.3 keV)')
ax.set_xlabel('Deuteron Energy (keV)'); ax.set_ylabel(f'Counts / {width_e:.2f} keV')
ax.set_title('Deuteron Energy Spectrum')
ax.legend(); ax.grid(True, linestyle='--', alpha=0.4); ax.minorticks_on()
plt.tight_layout(); plt.savefig('fig3_deuteron_energy.png', dpi=300); plt.show()

# ---- Figure 4: Time vs Radial Position ----
radii = np.sqrt(capture_positions[:,0]**2 + capture_positions[:,1]**2)
fig, ax = plt.subplots()
sc = ax.scatter(radii, capture_times, c=capture_energies, cmap='plasma',
                s=15, alpha=0.6, edgecolors='black', linewidth=0.3)
ax.axvline(33.5, color='blue', linestyle='--', alpha=0.7, label='D₂O boundary')
ax.set_xlabel('Radial Position (cm)'); ax.set_ylabel('Capture Time (μs)')
ax.set_title('Time vs Radial Position')
ax.set_xlim(0,60)
cbar = plt.colorbar(sc); cbar.set_label('Electron Energy (MeV)')
ax.legend(); ax.grid(True, linestyle='--', alpha=0.4); ax.minorticks_on()
plt.tight_layout(); plt.savefig('fig4_time_vs_radius.png', dpi=300); plt.show()

print("\nAll figures saved successfully.")

Total particles: 52,859,264
Building lookup tables...
Deuterons found: 821,476
Gamma parents (neutrons): 1,418
Electron parents (gammas): 2,842
Extracting capture electrons...


: 

In [ ]:
#!/usr/bin/env python
import uproot
import numpy as np
import matplotlib.pyplot as plt
import awkward as ak
from matplotlib.patches import Circle, Patch
from scipy.optimize import curve_fit
from collections import defaultdict

# ============================================================
# THESIS-STYLE PLOT SETTINGS
# ============================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 16,
    "axes.titlesize": 18,
    "legend.fontsize": 11,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "axes.linewidth": 1.2,
    "lines.linewidth": 1.8,
    "figure.figsize": (12.8, 6.2),
    "figure.dpi": 140,
})

DEUTERON_PDG = 1000010020
GAMMA_PDG = 22
ELECTRON_PDG = 11

# ============================================================
# LOAD DATA
# ============================================================
path = "Sim_D2ODetector003.root"
file = uproot.open(path)
event_data = file["Sim_Tree"]["eventData"]

all_pdg = event_data["allParticlePDG"].array()
all_track = event_data["allParticleTrackID"].array()
all_parent = event_data["allParticleParentID"].array()
all_time = event_data["allParticleTime"].array()
all_energy = event_data["allParticleEnergy"].array()
all_posX = event_data["allParticlePosX"].array()
all_posY = event_data["allParticlePosY"].array()
all_posZ = event_data["allParticlePosZ"].array()

pdg = ak.to_numpy(ak.flatten(all_pdg))
track = ak.to_numpy(ak.flatten(all_track))
parent = ak.to_numpy(ak.flatten(all_parent))
time = ak.to_numpy(ak.flatten(all_time))
energy = ak.to_numpy(ak.flatten(all_energy))
posX = ak.to_numpy(ak.flatten(all_posX))
posY = ak.to_numpy(ak.flatten(all_posY))
posZ = ak.to_numpy(ak.flatten(all_posZ))

print(f"Total particles: {len(pdg):,}")

# ============================================================
# FIND CAPTURE GAMMAS (2.22 MeV)
# ============================================================
print("Finding capture gammas (2.22 MeV)...")

# Filter for gammas with energy near 2.22 MeV
gamma_mask = (pdg == GAMMA_PDG) & (energy > 2.0) & (energy < 2.5)
gamma_indices = np.where(gamma_mask)[0]
print(f"Capture gammas (2.22 MeV): {len(gamma_indices):,}")

if len(gamma_indices) == 0:
    raise SystemExit("No capture gammas found. Check simulation or energy cut.")

# ============================================================
# BUILD GAMMA -> ELECTRON LOOKUP (fast)
# ============================================================
print("Building gamma -> electron lookup...")

gamma_to_electrons = defaultdict(list)
for i, (p, par) in enumerate(zip(pdg, parent)):
    if p == ELECTRON_PDG:
        gamma_to_electrons[par].append(i)

print(f"Gamma parents with electrons: {len(gamma_to_electrons):,}")

# ============================================================
# EXTRACT COMPTON ELECTRONS
# ============================================================
print("Extracting Compton electrons...")

capture_times = []       # μs
capture_energies = []    # MeV
capture_positions = []   # (x,y,z) cm
capture_gamma_times = [] # μs

for g_idx in gamma_indices:
    gamma_track = track[g_idx]
    gamma_time = time[g_idx] / 1000.0  # ns -> μs
    
    for e_idx in gamma_to_electrons.get(gamma_track, []):
        e_energy = energy[e_idx]
        e_time = time[e_idx] / 1000.0
        # Only keep Compton electrons (energy > 0.1 MeV)
        # and creation time similar to gamma (within ~ns)
        if e_energy > 0.1 and abs(e_time - gamma_time) < 0.01:  # within 10 ns
            capture_times.append(e_time)
            capture_energies.append(e_energy)
            capture_positions.append((posX[e_idx], posY[e_idx], posZ[e_idx]))
            capture_gamma_times.append(gamma_time)

capture_times = np.array(capture_times)
capture_energies = np.array(capture_energies)
capture_positions = np.array(capture_positions)
capture_gamma_times = np.array(capture_gamma_times)

print(f"Compton electrons found: {len(capture_times):,}")
print(f"Fraction: {100 * len(capture_times) / max(1, len(gamma_indices)):.1f}% of gammas produced electrons")

if len(capture_times) == 0:
    raise SystemExit("No Compton electrons found. Try lowering the energy threshold.")

# ============================================================
# EXPONENTIAL DECAY FIT
# ============================================================
def exp_decay(t, A, tau, C):
    return A * np.exp(-t / tau) + C

# ============================================================
# PLOT 1: XY POSITIONS
# ============================================================
fig, ax = plt.subplots()
sc = ax.scatter(capture_positions[:,0], capture_positions[:,1],
                c=capture_times, cmap='viridis', s=15, alpha=0.7,
                edgecolors='black', linewidth=0.3)

boundaries = [(33.5, 'D₂O', 'blue'), (42.5, 'Acrylic', 'magenta'),
              (48.5, 'Tyvek', 'gray'), (52.5, 'Steel', 'darkgray')]
for r, label, col in boundaries:
    circle = Circle((0,0), r, fill=False, edgecolor=col, linestyle='--', linewidth=1.5)
    ax.add_patch(circle)

ax.set_xlabel('X Position (cm)', fontweight='bold')
ax.set_ylabel('Y Position (cm)', fontweight='bold')
ax.set_title(f'Neutron Capture Positions (XY)', pad=15, fontweight='bold')
ax.set_aspect('equal'); ax.set_xlim(-60,60); ax.set_ylim(-60,60)
cbar = plt.colorbar(sc); cbar.set_label('Capture Time (μs)', fontsize=12)

legend_elements = [Patch(facecolor='none', edgecolor=c, linestyle='--', linewidth=2, label=l)
                   for _, l, c in boundaries]
ax.legend(handles=legend_elements, loc='upper right', frameon=True,
          framealpha=0.96, fancybox=False, edgecolor='black')
ax.grid(True, linestyle='--', alpha=0.4); ax.minorticks_on()
ax.tick_params(which='both', direction='in', top=True, right=True)
ax.tick_params(which='major', length=6, width=1.2)
ax.tick_params(which='minor', length=3, width=0.8)
plt.tight_layout(); plt.savefig('fig1_capture_positions.png', dpi=300, bbox_inches='tight'); plt.close()
print("✓ Saved fig1_capture_positions.png")

# ============================================================
# PLOT 2: CAPTURE TIME DISTRIBUTION + EXPONENTIAL FIT
# ============================================================
fig, ax = plt.subplots()
bins = np.linspace(0, 500, 100)
hist, edges = np.histogram(capture_times, bins=bins)
centers = (edges[:-1] + edges[1:]) / 2
width = edges[1] - edges[0]
norm = hist / width

ax.step(centers, norm, where='mid', color='#1f77b4', linewidth=2, label=f'Data (n={len(capture_times):,})')
ax.plot(centers, norm, 'o', color='#1f77b4', markersize=3,
        markerfacecolor='white', markeredgecolor='#1f77b4')

# Fit
mask = (centers > 50) & (norm > 0)
if np.sum(mask) > 5:
    popt, _ = curve_fit(exp_decay, centers[mask], norm[mask], p0=[max(norm), 100, 0])
    x_fit = np.linspace(0, 500, 200)
    ax.plot(x_fit, exp_decay(x_fit, *popt), 'r-', linewidth=2,
            label=f'Fit: τ = {popt[1]:.1f} μs')
    print(f"Fit result: τ = {popt[1]:.1f} μs")

mean_t = np.mean(capture_times); median_t = np.median(capture_times)
ax.axvline(mean_t, color='red', linestyle='--', alpha=0.7, label=f'Mean: {mean_t:.1f} μs')
ax.axvline(median_t, color='green', linestyle='--', alpha=0.7, label=f'Median: {median_t:.1f} μs')

ax.set_xlabel('Capture Time (μs)', fontweight='bold')
ax.set_ylabel(f'Counts / {width:.1f} μs', fontweight='bold')
ax.set_title('Neutron Capture Time Distribution (Compton Electrons)', pad=15, fontweight='bold')
ax.set_xlim(0, 500); ax.set_yscale('log')
ax.legend(frameon=True, framealpha=0.96, fancybox=False, edgecolor='black')
ax.grid(True, linestyle='--', alpha=0.4); ax.minorticks_on()
ax.tick_params(which='both', direction='in', top=True, right=True)
plt.tight_layout(); plt.savefig('fig2_capture_time_fit.png', dpi=300, bbox_inches='tight'); plt.close()
print("✓ Saved fig2_capture_time_fit.png")

# ============================================================
# PLOT 3: DEUTERON ENERGY SPECTRUM (for verification)
# ============================================================
is_deuteron = (pdg == DEUTERON_PDG)
deuteron_energies_keV = energy[is_deuteron] * 1000

fig, ax = plt.subplots()
bins_e = np.linspace(0, 2.0, 50)
hist_e, edges_e = np.histogram(deuteron_energies_keV, bins=bins_e)
centers_e = (edges_e[:-1] + edges_e[1:]) / 2
width_e = edges_e[1] - edges_e[0]
norm_e = hist_e / width_e

ax.step(centers_e, norm_e, where='mid', color='#9467bd', linewidth=2,
        label=f'Deuterons (n={len(deuteron_energies_keV):,})')
ax.plot(centers_e, norm_e, 'o', color='#9467bd', markersize=3,
        markerfacecolor='white', markeredgecolor='#9467bd')
ax.axvline(1.3, color='red', linestyle='--', linewidth=2, label='Expected (1.3 keV)')

ax.set_xlabel('Deuteron Energy (keV)', fontweight='bold')
ax.set_ylabel(f'Counts / {width_e:.2f} keV', fontweight='bold')
ax.set_title('Deuteron Energy Spectrum (n + H → D + γ)', pad=15, fontweight='bold')
ax.legend(frameon=True, framealpha=0.96, fancybox=False, edgecolor='black')
ax.grid(True, linestyle='--', alpha=0.4); ax.minorticks_on()
ax.tick_params(which='both', direction='in', top=True, right=True)
plt.tight_layout(); plt.savefig('fig3_deuteron_energy.png', dpi=300, bbox_inches='tight'); plt.close()
print("✓ Saved fig3_deuteron_energy.png")

# ============================================================
# PLOT 4: TIME VS RADIAL POSITION
# ============================================================
radii = np.sqrt(capture_positions[:,0]**2 + capture_positions[:,1]**2)

fig, ax = plt.subplots()
sc = ax.scatter(radii, capture_times, c=capture_energies, cmap='plasma',
                s=15, alpha=0.6, edgecolors='black', linewidth=0.3)
ax.axvline(33.5, color='blue', linestyle='--', alpha=0.7, label='D₂O boundary')
ax.axvline(42.5, color='magenta', linestyle='--', alpha=0.7, label='Acrylic boundary')

ax.set_xlabel('Radial Position (cm)', fontweight='bold')
ax.set_ylabel('Capture Time (μs)', fontweight='bold')
ax.set_title('Capture Time vs Radial Position', pad=15, fontweight='bold')
ax.set_xlim(0, 60)
cbar = plt.colorbar(sc); cbar.set_label('Electron Energy (MeV)', fontsize=12)
ax.legend(frameon=True, framealpha=0.96, fancybox=False, edgecolor='black')
ax.grid(True, linestyle='--', alpha=0.4); ax.minorticks_on()
ax.tick_params(which='both', direction='in', top=True, right=True)
plt.tight_layout(); plt.savefig('fig4_time_vs_radius.png', dpi=300, bbox_inches='tight'); plt.close()
print("✓ Saved fig4_time_vs_radius.png")

# ============================================================
# SUMMARY
# ============================================================
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"Capture gammas (2.22 MeV):        {len(gamma_indices):,}")
print(f"Compton electrons found:          {len(capture_times):,}")
print(f"Fraction of gammas producing e⁻:  {100 * len(capture_times) / max(1, len(gamma_indices)):.1f}%")
print(f"Mean capture time:                {np.mean(capture_times):.2f} μs")
print(f"Median capture time:              {np.median(capture_times):.2f} μs")
print("\nFigures saved:")
print("  - fig1_capture_positions.png")
print("  - fig2_capture_time_fit.png")
print("  - fig3_deuteron_energy.png")
print("  - fig4_time_vs_radius.png")
print("="*80)

Total particles: 52,859,264
Finding capture gammas (2.22 MeV)...
Capture gammas (2.22 MeV): 840,031
Building gamma -> electron lookup...
Gamma parents with electrons: 2,842
Extracting Compton electrons...


In [ ]:
# %% [markdown]
# # Neutron Capture Time Analysis
# 
# This notebook analyzes deuteron and tritium creation times from neutron capture events.
# 
# **Physics:**
# - n + H → D + γ (Hydrogen capture, 2.22 MeV)
# - n + D → T + γ (Deuterium capture, 6.25 MeV)
# 
# **Expected Results:**
# - Deuterons (H₂O): ~200 μs capture time
# - Tritium (D₂O): ~2-5 ms capture time

# %% [markdown]
# ## 1. Import Libraries

# %%
import uproot
import numpy as np
import matplotlib.pyplot as plt
import awkward as ak
import os
from scipy.optimize import curve_fit

print("Libraries imported successfully")

# %% [markdown]
# ## 2. Configuration

# %%
# ============================================================
# CONFIGURATION - ADJUST THESE PARAMETERS
# ============================================================

# Binning settings
N_BINS_DEUTERON = 5000      # Number of bins for deuteron plot
N_BINS_TRITIUM = 300        # Number of bins for tritium plot
N_BINS_OVERLAY = 300        # Number of bins for overlay plot
N_BINS_FULL_RANGE = 300     # Number of bins for full range plot

# Time range settings
BIN_RANGE_MIN = 16          # Minimum time range for plots (μs)
BIN_RANGE_MAX = 1000        # Maximum time range for deuteron plots (μs)

# Plot settings
PLOT_STYLE = 'seaborn-v0_8-darkgrid'
FIGURE_SIZE = (12, 8)
FONT_SIZE = 12

print("Configuration loaded")

# %% [markdown]
# ## 3. Exponential Fit Function

# %%
def exponential_decay(t, A, tau, C):
    """
    Exponential decay function: A * exp(-t/tau) + C
    
    Parameters:
    -----------
    t : array_like
        Time values
    A : float
        Amplitude
    tau : float
        Decay constant (neutron lifetime)
    C : float
        Constant offset
    
    Returns:
    --------
    array_like : Fitted values
    """
    return A * np.exp(-t / tau) + C

print("Exponential decay function defined")

# %% [markdown]
# ## 4. Load ROOT File

# %%
# ============================================================
# SPECIFY YOUR FILE HERE
# ============================================================
path = "Sim_D2ODetector003.root"  # <-- CHANGE THIS TO YOUR FILE

if not os.path.exists(path):
    print(f"Error: {path} not found!")
    print("Please check the file path and name.")
    exit()

# Open the ROOT file
file = uproot.open(path)
Sim_Tree = file["Sim_Tree"]
event_data = Sim_Tree["eventData"]

print(f"File opened: {path}")
print(f"Total events: {Sim_Tree.num_entries}")

# %% [markdown]
# ## 5. Load Particle Data

# %%
print("\nLoading particle data...")

try:
    all_particle_pdg = event_data["allParticlePDG"].array()
    all_particle_time = event_data["allParticleTime"].array()
    all_particle_energy = event_data["allParticleEnergy"].array()
    all_particle_name = event_data["allParticleName"].array()
    has_all_particles = True
    print("✓ All particles data found")
except Exception as e:
    has_all_particles = False
    print(f"✗ All particles data not found: {e}")
    print("Please make sure your ROOT file contains allParticle branches")
    exit()

# %%
# Flatten the data for easier access
all_pdg_flat = ak.to_numpy(ak.flatten(all_particle_pdg))
all_time_flat = ak.to_numpy(ak.flatten(all_particle_time))
all_energy_flat = ak.to_numpy(ak.flatten(all_particle_energy))
all_name_flat = ak.to_numpy(ak.flatten(all_particle_name))

print(f"Total particles: {len(all_pdg_flat):,}")

# %% [markdown]
# ## 6. Identify Deuterons and Tritium

# %%
# ============================================================
# PDG CODES FOR CAPTURE PRODUCTS
# ============================================================
deuteron_pdg = 1000010020    # Deuteron from n + H → D + γ
tritium_pdg = 1000010030     # Tritium from n + D → T + γ

# Find deuterons and tritium
is_deuteron = (all_pdg_flat == deuteron_pdg)
is_tritium = (all_pdg_flat == tritium_pdg)

n_deuterons = np.sum(is_deuteron)
n_tritium = np.sum(is_tritium)

print(f"\n{'='*60}")
print("PARTICLE IDENTIFICATION RESULTS")
print(f"{'='*60}")
print(f"Deuterons (n + H → D + γ): {n_deuterons:,}")
print(f"Tritium (n + D → T + γ):   {n_tritium:,}")
print(f"Total capture products:    {n_deuterons + n_tritium:,}")

if n_deuterons == 0 and n_tritium == 0:
    print("\n⚠️ No deuterons or tritium found!")
    print("Possible reasons:")
    print("  1. No neutron captures occurred in your simulation")
    print("  2. Check NeutronTreatment = 1 in beamOn.dat")
    print("  3. Run more events")
    exit()

# %% [markdown]
# ## 7. Extract Creation Times

# %%
# Convert times to microseconds (μs)
deuteron_times_us = all_time_flat[is_deuteron] / 1000 if n_deuterons > 0 else np.array([])
tritium_times_us = all_time_flat[is_tritium] / 1000 if n_tritium > 0 else np.array([])

# Get energies
deuteron_energies = all_energy_flat[is_deuteron] * 1000 if n_deuterons > 0 else np.array([])  # keV
tritium_energies = all_energy_flat[is_tritium] * 1000 if n_tritium > 0 else np.array([])      # keV

print(f"\n{'='*60}")
print("CREATION TIME STATISTICS")
print(f"{'='*60}")

if n_deuterons > 0:
    print(f"\nDeuterons ({n_deuterons:,} particles):")
    print(f"  Mean time:   {np.mean(deuteron_times_us):.2f} μs")
    print(f"  Median time: {np.median(deuteron_times_us):.2f} μs")
    print(f"  Min time:    {np.min(deuteron_times_us):.2f} μs")
    print(f"  Max time:    {np.max(deuteron_times_us):.2f} μs")
    print(f"  Mean energy: {np.mean(deuteron_energies):.2f} keV")

if n_tritium > 0:
    print(f"\nTritium ({n_tritium:,} particles):")
    print(f"  Mean time:   {np.mean(tritium_times_us):.2f} μs ({np.mean(tritium_times_us)/1000:.3f} ms)")
    print(f"  Median time: {np.median(tritium_times_us):.2f} μs")
    print(f"  Min time:    {np.min(tritium_times_us):.2f} μs")
    print(f"  Max time:    {np.max(tritium_times_us):.2f} μs")
    print(f"  Mean energy: {np.mean(tritium_energies):.2f} keV")

# %% [markdown]
# ## 8. Plot 1: Deuteron Creation Time

# %%
fig, ax = plt.subplots(figsize=FIGURE_SIZE)

if n_deuterons > 0:
    # Define time range for deuterons (start from 16 μs to avoid prompt events)
    t_min = BIN_RANGE_MIN
    t_max = BIN_RANGE_MAX
    
    mask_plot = (deuteron_times_us >= t_min) & (deuteron_times_us <= t_max)
    deuteron_times_plot = deuteron_times_us[mask_plot]
    
    if len(deuteron_times_plot) > 0:
        # Histogram
        hist, bin_edges = np.histogram(deuteron_times_plot, bins=N_BINS_DEUTERON, range=(t_min, t_max))
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        bin_width = bin_edges[1] - bin_edges[0]
        
        # Normalize by bin width
        hist_norm = hist / bin_width
        
        # Plot as step histogram
        ax.step(bin_edges[:-1], hist_norm, where='post', color='#2ca02c', linewidth=2, label='Data')
        
        # Exponential fit
        mask_fit = hist > 0
        bin_centers_fit = bin_centers[mask_fit]
        hist_fit = hist[mask_fit] / bin_width
        
        if len(bin_centers_fit) > 3:
            try:
                p0 = [max(hist_fit), 100, 0]
                popt, pcov = curve_fit(exponential_decay, bin_centers_fit, hist_fit, p0=p0, maxfev=5000)
                perr = np.sqrt(np.diag(pcov))
                
                x_fit = np.linspace(t_min, t_max, 200)
                y_fit = exponential_decay(x_fit, *popt)
                ax.plot(x_fit, y_fit, 'r-', linewidth=2, 
                        label=f'Exponential Fit: τ = {popt[1]:.1f} ± {perr[1]:.1f} μs')
                
                print(f"\nDeuteron fit:")
                print(f"  τ = {popt[1]:.1f} ± {perr[1]:.1f} μs")
                print(f"  A = {popt[0]:.2f} ± {perr[0]:.2f}")
                print(f"  C = {popt[2]:.2f} ± {perr[2]:.2f}")
            except Exception as e:
                print(f"Fit failed: {e}")
        
        ax.set_xlabel('Creation Time (μs)', fontsize=14)
        ax.set_ylabel(f'Counts / {bin_width:.2f} μs', fontsize=14)
        ax.set_title(f'Deuteron Creation Time (n={len(deuteron_times_plot):,})', fontsize=16, fontweight='bold')
        ax.set_xlim(t_min, t_max)
        ax.legend(fontsize=12)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'No Deuterons in {t_min}-{t_max} μs range', 
                transform=ax.transAxes, ha='center', va='center', fontsize=14)
else:
    ax.text(0.5, 0.5, 'No Deuterons Found', transform=ax.transAxes, ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.savefig('figure_deuteron.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: figure_deuteron.png")

# %% [markdown]
# ## 9. Plot 2: Tritium Creation Time

# %%
fig, ax = plt.subplots(figsize=FIGURE_SIZE)

if n_tritium > 0:
    # Define time range for tritium (start from 16 μs)
    t_min = BIN_RANGE_MIN
    t_max = np.max(tritium_times_us)
    
    mask_plot = (tritium_times_us >= t_min)
    tritium_times_plot = tritium_times_us[mask_plot]
    
    if len(tritium_times_plot) > 0:
        # Histogram
        hist, bin_edges = np.histogram(tritium_times_plot, bins=N_BINS_TRITIUM, range=(t_min, t_max))
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        bin_width = bin_edges[1] - bin_edges[0]
        
        # Normalize by bin width
        hist_norm = hist / bin_width
        
        # Plot as step histogram
        ax.step(bin_edges[:-1], hist_norm, where='post', color='#ff7f0e', linewidth=2, label='Data')
        
        # Exponential fit
        mask_fit = hist > 0
        bin_centers_fit = bin_centers[mask_fit]
        hist_fit = hist[mask_fit] / bin_width
        
        if len(bin_centers_fit) > 3:
            try:
                p0 = [max(hist_fit), np.mean(tritium_times_plot), 0]
                popt, pcov = curve_fit(exponential_decay, bin_centers_fit, hist_fit, p0=p0, maxfev=5000)
                perr = np.sqrt(np.diag(pcov))
                
                x_fit = np.linspace(t_min, t_max, 200)
                y_fit = exponential_decay(x_fit, *popt)
                ax.plot(x_fit, y_fit, 'r-', linewidth=2, 
                        label=f'Exponential Fit: τ = {popt[1]:.1f} ± {perr[1]:.1f} μs')
                
                print(f"\nTritium fit:")
                print(f"  τ = {popt[1]:.1f} ± {perr[1]:.1f} μs")
                print(f"  A = {popt[0]:.2f} ± {perr[0]:.2f}")
                print(f"  C = {popt[2]:.2f} ± {perr[2]:.2f}")
            except Exception as e:
                print(f"Fit failed: {e}")
        
        ax.set_xlabel('Creation Time (μs)', fontsize=14)
        ax.set_ylabel(f'Counts / {bin_width:.2f} μs', fontsize=14)
        ax.set_title(f'Tritium Creation Time (n={len(tritium_times_plot):,})', fontsize=16, fontweight='bold')
        ax.legend(fontsize=12)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'No Tritium above {t_min} μs', 
                transform=ax.transAxes, ha='center', va='center', fontsize=14)
else:
    ax.text(0.5, 0.5, 'No Tritium Found', transform=ax.transAxes, ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.savefig('figure_tritium.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: figure_tritium.png")

# %% [markdown]
# ## 10. Plot 3: Deuteron vs Tritium Overlay

# %%
fig, ax = plt.subplots(figsize=FIGURE_SIZE)

# Store fit results for summary
deuteron_tau = None
deuteron_tau_err = None
tritium_tau = None
tritium_tau_err = None

# Deuterons (16-1000 μs)
if n_deuterons > 0:
    t_min_d = BIN_RANGE_MIN
    t_max_d = BIN_RANGE_MAX
    mask_d = (deuteron_times_us >= t_min_d) & (deuteron_times_us <= t_max_d)
    deuteron_times_plot = deuteron_times_us[mask_d]
    
    if len(deuteron_times_plot) > 0:
        hist_d, bin_edges_d = np.histogram(deuteron_times_plot, bins=N_BINS_OVERLAY, range=(t_min_d, t_max_d))
        bin_width_d = bin_edges_d[1] - bin_edges_d[0]
        hist_d_norm = hist_d / bin_width_d
        
        ax.step(bin_edges_d[:-1], hist_d_norm, where='post', color='#2ca02c', linewidth=2, 
                label=f'Deuterons (n={len(deuteron_times_plot):,})')
        
        # Fit deuterons
        mask_fit_d = hist_d > 0
        bin_centers_fit_d = (bin_edges_d[:-1] + bin_edges_d[1:])[mask_fit_d] / 2
        hist_fit_d = hist_d[mask_fit_d] / bin_width_d
        
        if len(bin_centers_fit_d) > 3:
            try:
                p0 = [max(hist_fit_d), 100, 0]
                popt_d, pcov_d = curve_fit(exponential_decay, bin_centers_fit_d, hist_fit_d, p0=p0, maxfev=5000)
                perr_d = np.sqrt(np.diag(pcov_d))
                deuteron_tau = popt_d[1]
                deuteron_tau_err = perr_d[1]
                
                x_fit = np.linspace(t_min_d, t_max_d, 200)
                y_fit = exponential_decay(x_fit, *popt_d)
                ax.plot(x_fit, y_fit, 'g-', linewidth=2, 
                        label=f'Deuterons fit: τ = {popt_d[1]:.1f} ± {perr_d[1]:.1f} μs')
            except:
                pass

# Tritium (16 μs to max)
if n_tritium > 0:
    t_min_t = BIN_RANGE_MIN
    mask_t = (tritium_times_us >= t_min_t)
    tritium_times_plot = tritium_times_us[mask_t]
    
    if len(tritium_times_plot) > 0:
        t_max_t = np.max(tritium_times_plot)
        hist_t, bin_edges_t = np.histogram(tritium_times_plot, bins=N_BINS_OVERLAY, range=(t_min_t, t_max_t))
        bin_width_t = bin_edges_t[1] - bin_edges_t[0]
        hist_t_norm = hist_t / bin_width_t
        
        ax.step(bin_edges_t[:-1], hist_t_norm, where='post', color='#ff7f0e', linewidth=2, 
                label=f'Tritium (n={len(tritium_times_plot):,})')
        
        # Fit tritium
        mask_fit_t = hist_t > 0
        bin_centers_fit_t = (bin_edges_t[:-1] + bin_edges_t[1:])[mask_fit_t] / 2
        hist_fit_t = hist_t[mask_fit_t] / bin_width_t
        
        if len(bin_centers_fit_t) > 3:
            try:
                p0 = [max(hist_fit_t), np.mean(tritium_times_plot), 0]
                popt_t, pcov_t = curve_fit(exponential_decay, bin_centers_fit_t, hist_fit_t, p0=p0, maxfev=5000)
                perr_t = np.sqrt(np.diag(pcov_t))
                tritium_tau = popt_t[1]
                tritium_tau_err = perr_t[1]
                
                x_fit = np.linspace(t_min_t, t_max_t, 200)
                y_fit = exponential_decay(x_fit, *popt_t)
                ax.plot(x_fit, y_fit, 'r-', linewidth=2, 
                        label=f'Tritium fit: τ = {popt_t[1]:.1f} ± {perr_t[1]:.1f} μs')
            except:
                pass

ax.set_xlabel('Creation Time (μs)', fontsize=14)
ax.set_ylabel(f'Counts / {bin_width_d:.2f} μs', fontsize=14)
ax.set_title('Deuteron vs Tritium Creation Time Comparison', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figure_deuteron_tritium_overlay.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: figure_deuteron_tritium_overlay.png")

# %% [markdown]
# ## 11. Plot 4: Tritium Full Range

# %%
if n_tritium > 0:
    fig, ax = plt.subplots(figsize=FIGURE_SIZE)
    
    # Full range histogram
    hist, bin_edges = np.histogram(tritium_times_us, bins=N_BINS_FULL_RANGE)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_width = bin_edges[1] - bin_edges[0]
    
    hist_norm = hist / bin_width
    
    ax.step(bin_edges[:-1], hist_norm, where='post', color='#ff7f0e', linewidth=2, label='Data')
    
    # Exponential fit
    mask_fit = hist > 0
    bin_centers_fit = bin_centers[mask_fit]
    hist_fit = hist[mask_fit] / bin_width
    
    if len(bin_centers_fit) > 3:
        try:
            p0 = [max(hist_fit), np.mean(tritium_times_us), 0]
            popt, pcov = curve_fit(exponential_decay, bin_centers_fit, hist_fit, p0=p0, maxfev=5000)
            perr = np.sqrt(np.diag(pcov))
            
            x_fit = np.linspace(min(bin_centers), max(bin_centers), 200)
            y_fit = exponential_decay(x_fit, *popt)
            ax.plot(x_fit, y_fit, 'r-', linewidth=2, 
                    label=f'Exponential Fit: τ = {popt[1]:.1f} ± {perr[1]:.1f} μs')
            
            print(f"\nTritium full range fit:")
            print(f"  τ = {popt[1]:.1f} ± {perr[1]:.1f} μs")
            print(f"  A = {popt[0]:.2f} ± {perr[0]:.2f}")
            print(f"  C = {popt[2]:.2f} ± {perr[2]:.2f}")
        except Exception as e:
            print(f"Fit failed: {e}")
    
    ax.set_xlabel('Creation Time (μs)', fontsize=14)
    ax.set_ylabel(f'Counts / {bin_width:.2f} μs', fontsize=14)
    ax.set_title(f'Tritium Creation Time - Full Range (n={n_tritium:,})', fontsize=16, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('figure_tritium_full_range.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Saved: figure_tritium_full_range.png")

# %% [markdown]
# ## 12. Summary Statistics

# %%
print("\n" + "="*80)
print("CREATION TIME STATISTICS")
print("="*80)

if n_deuterons > 0:
    print(f"\nDeuterons ({n_deuterons:,} particles):")
    print(f"  Mean time (full range):   {np.mean(deuteron_times_us):.2f} μs")
    print(f"  Median time (full range): {np.median(deuteron_times_us):.2f} μs")
    
    mask_range = (deuteron_times_us >= BIN_RANGE_MIN) & (deuteron_times_us <= BIN_RANGE_MAX)
    if np.sum(mask_range) > 0:
        range_times = deuteron_times_us[mask_range]
        print(f"\n  Statistics for {BIN_RANGE_MIN}-{BIN_RANGE_MAX} μs range (n={len(range_times)}):")
        print(f"    Mean time:   {np.mean(range_times):.2f} μs")
        print(f"    Median time: {np.median(range_times):.2f} μs")
    
    if deuteron_tau is not None:
        print(f"\n  Exponential fit (16-1000 μs):")
        print(f"    τ = {deuteron_tau:.1f} ± {deuteron_tau_err:.1f} μs")
        print(f"    Relative error: {100 * deuteron_tau_err / deuteron_tau:.1f}%")

if n_tritium > 0:
    print(f"\nTritium ({n_tritium:,} particles):")
    print(f"  Mean time (full range):   {np.mean(tritium_times_us):.2f} μs ({np.mean(tritium_times_us)/1000:.3f} ms)")
    print(f"  Median time (full range): {np.median(tritium_times_us):.2f} μs")
    
    mask_range = (tritium_times_us >= BIN_RANGE_MIN)
    if np.sum(mask_range) > 0:
        range_times = tritium_times_us[mask_range]
        print(f"\n  Statistics for ≥{BIN_RANGE_MIN} μs range (n={len(range_times)}):")
        print(f"    Mean time:   {np.mean(range_times):.2f} μs")
        print(f"    Median time: {np.median(range_times):.2f} μs")
    
    if tritium_tau is not None:
        print(f"\n  Exponential fit (≥16 μs):")
        print(f"    τ = {tritium_tau:.1f} ± {tritium_tau_err:.1f} μs")
        print(f"    Relative error: {100 * tritium_tau_err / tritium_tau:.1f}%")

# %% [markdown]
# ## 13. Summary Table

# %%
print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)
print(f"{'Metric':<45} {'Value':<30}")
print("-"*75)
print(f"{'Deuterons (n + H → D + γ)':<45} {n_deuterons:,}")
print(f"{'Tritium (n + D → T + γ)':<45} {n_tritium:,}")
print(f"{'Total Capture Products':<45} {n_deuterons + n_tritium:,}")

if n_deuterons + n_tritium > 0:
    print(f"{'Deuteron Fraction':<45} {100 * n_deuterons / (n_deuterons + n_tritium):.1f}%")
    print(f"{'Tritium Fraction':<45} {100 * n_tritium / (n_deuterons + n_tritium):.1f}%")

if deuteron_tau is not None:
    print(f"{'Deuteron lifetime (τ)':<45} {deuteron_tau:.1f} ± {deuteron_tau_err:.1f} μs")
if tritium_tau is not None:
    print(f"{'Tritium lifetime (τ)':<45} {tritium_tau:.1f} ± {tritium_tau_err:.1f} μs")

print("\n" + "="*80)
print("FILES GENERATED")
print("="*80)
print("  - figure_deuteron.png")
print("  - figure_tritium.png")
print("  - figure_deuteron_tritium_overlay.png")
if n_tritium > 0:
    print("  - figure_tritium_full_range.png")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)